In [47]:
import pandas as pd
from sentence_transformers import SentenceTransformer

In [79]:
from qdrant_client import models, QdrantClient
 

In [53]:
from openai import OpenAI

# loading  dataset

In [2]:
wineDataset=pd.read_csv('top_rated_wines.csv')

In [9]:
wineDataset=wineDataset.fillna('')

In [11]:
wineDatasetList=wineDataset.to_dict('records')
print('wineDatasetList',wineDatasetList[0])

wineDatasetList {'name': '3 Rings Reserve Shiraz 2004', 'region': 'Barossa Valley, Barossa, South Australia, Australia', 'variety': 'Red Wine', 'rating': 96.0, 'notes': 'Vintage Comments : Classic Barossa vintage conditions. An average wet Spring followed by extreme heat in early February. Occasional rainfall events kept the vines in good balance up to harvest in late March 2004. Very good quality coupled with good average yields. More than 30 months in wood followed by six months tank maturation of the blend prior to bottling, July 2007. '}


#  Prepare Vector DB

In [ ]:
collection_name="top_wines"

In [ ]:
# Create the embedding encoder
encoder = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
 

In [ ]:
# create dataset by encoding the wine notes
dataset=[]
for idx,doc in enumerate(wineDatasetList):
    dataset.append(models.PointStruct(
        id=idx,
        vector=encoder.embeddings.create(input=doc["notes"],model="text-embedding-nomic-embed-text-v1.5").data[0].embedding,
        payload=doc
    ))
  

In [80]:

 # Create in-memory Qdrant instance
qdrant = QdrantClient(":memory:")

qdrant.recreate_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=len(dataset[0].vector), # Vector size is defined by used model (768 for text-embedding-nomic-embed-text-v1.5)
        distance=models.Distance.COSINE
    )
)


C:\Users\syedfurx\AppData\Local\Temp\ipykernel_28104\593163456.py:4: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


True

In [81]:

# vectorize!
qdrant.upload_points(
    collection_name=collection_name,
    points=dataset
)

# Search vector data base

In [82]:
user_prompt = "Suggest me an amazing Malbec wine from Argentina"

In [69]:
# convert user query to embeddings 
query_vector = encoder.embeddings.create(input=user_prompt,model="text-embedding-nomic-embed-text-v1.5").data[0].embedding

In [ ]:
#searching with vecotr db to find similar vecotrs
hits = qdrant.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=3
)
 

In [147]:
fullList=[]
for point,scoredPoint in hits:
    
    val=scoredPoint[0].payload
    fullList.append(val)
    print('Name',val.get('name'))
    print('region',val.get('region'))
    print('variety',val.get('variety'))
    print('rating',val.get('rating'))
    print('notes',val.get('notes'))
   
    

Name Catena Zapata Adrianna Vineyard Malbec 2004
region Argentina
variety Red Wine
rating 97.0
notes "The single-vineyard 2004 Malbec Adrianna Vineyard from the Gualtallary district is inky purple with aromas of wood smoke, pencil lead, game, black cherry, and blackberry liqueur. Opulent, full-flavored, yet remarkably light on its feet, this medium to full-bodied Malbec is all about pleasure. It will certainly evolve for a decade but is hard to resist now. It is a fine test of one's ability to defer immediate gratification. When all is said and done, Catena Zapata is the Argentina winery of reference – the standard of excellence for comparing all others. The brilliant, forward-thinking Nicolas Catena remains in charge, with his daughter, Laura, playing an increasingly large role. The Catena Zapata winery is an essential destination for fans of both architecture and wine in Mendoza. It is hard to believe, given the surge in popularity of Malbec in recent years, that Catena Zapata only b

# check same question with LLM

In [141]:
# Create the embedding encoder
model = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

In [143]:
completion =model.chat.completions.create(
    model="qwen2.5-coder-3b-instruct",

    messages=[
        {"role": "system", "content": "You are chatbot, a wine specialist. Your top priority is to help guide users into selecting amazing wine and guide them with their requests."},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": "Here is my wine recommendation:"}
    ]
)

In [146]:
# result from LLM without embedding
print(completion.choices[0].message.content)

 The Château du Sud Malbec 2016.

This wine offers a deep purple color, with aromas of black fruit such as plums and blackberries. Its complex flavors include hints of vanilla, chocolate, and spice. The wine has good structure and tannins, making it a perfect choice for those who prefer full-bodied wines. 

The Château du Sud Malbec 2016 is produced in the Mendoza region of Argentina, known for its rich and diverse terroir. This bottle is guaranteed to impress with its flavor and complexity.

I hope you enjoy this wine!


# Same Question with Emabbedings

In [167]:
print(str(fullList[0]['name']))

Catena Zapata Adrianna Vineyard Malbec 2004


In [174]:
completion =model.chat.completions.create(
    model="mistralai/ministral-3-3b",

    messages=[
        {"role": "system", "content": "You are chatbot, a wine specialist. Your top priority is to help guide users into selecting amazing wine and guide them with their requests."},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": str(fullList)}
    ]
)

In [175]:
# result from LLM with embedding
print(completion.choices[0].message.content)

 Here’s a fantastic recommendation for you:

**Catena Zapata Adrianna Vineyard Malbec 2004**
- **Region**: Mendoza (Argentina)
- **Style**: Rich, full-bodied red wine with deep fruit flavors and velvety texture.
- **Why it’s amazing**:
  - This is one of the most celebrated Malbecs from Argentina, aged in French oak barrels to enhance its complexity. The 2004 vintage is particularly rare and sought-after, offering layers of blackberry, plum, and spice with a touch of earthy depth.
  - It’s a benchmark for Argentine Malbec, balancing power with elegance.

**Pairing Suggestion**: Serve this with grilled lamb chops, aged balsamic-glazed steak, or rich chocolate desserts (like dark chocolate fondant).

---
### **Alternative Options**:
1. **Penfolds Bin 20 (Malbec) 2015**
   - A bold, structured Malbec from Australia’s Adelaide Hills with notes of blackberry jam, tobacco, and a hint of cedar.

2. **Trapiche Malbec "El Poblado" 2018**
   - A more affordable yet high-quality option with ripe 